In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve
from numpy.polynomial.legendre import leggauss
import torch
from scipy.linalg import eigh
from scipy.linalg import solve



In [ ]:
#@title FEM in 1D

#Creates a mesh for the Domain, just for one Dimension.
def mesh(M,OMEGA):
  l_boundary, r_boundary = OMEGA
  triangles = np.linspace(l_boundary, r_boundary, M+1)
  return triangles

#Evaluates a function at every point in your mesh
def evaluate_function(triangles, function):
    inner_triangles = triangles[1:-1]
    function_values = np.array([])
    for i in range(len(triangles)):
        function_value = function(triangles[i])
        function_values.append(function_value)
    return function_values

# assembles the stiffness- and mass-matrix for 1D linear FEM discretization for A = - Laplace
def matrix(M, OMEGA, inner_triangles):
  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M
  stiff_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))
  mass_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))

  for i in range(stiff_matrix.shape[0]):
    stiff_matrix[i,i] = 2
    mass_matrix[i,i] = 4

  for i in range(stiff_matrix.shape[0]-1):
    stiff_matrix[i,i+1] = -1
    stiff_matrix[i+1,i] = -1
    mass_matrix[i,i+1] = 1
    mass_matrix[i+1,i] = 1

  stiff = 1/h * stiff_matrix
  mass = h/6 * mass_matrix

  return stiff, mass

# assembles the mass matrix
def mass(M, OMEGA, inner_triangles):
  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M
  mass_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))

  for i in range(mass_matrix.shape[0]):
    mass_matrix[i,i] = 4

  for i in range(mass_matrix.shape[0]-1):
    mass_matrix[i,i+1] = 1
    mass_matrix[i+1,i] = 1

  mass = h/6 * mass_matrix

  return mass

# calculates the right hand side vector, using a midpoint quadrature
def calculate_RHS(f, func, M, OMEGA, t):

  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M

  tri = func(M, OMEGA)
  RHS = np.zeros(M+1)

  for i in range(M):
    Integral_fphik = (1/2) * (tri[i+1] - tri[i]) * f(tri[i] + (tri[i+1] - tri[i])*(1/2), t)  #Q(f) = abs(T)*f(x_s)
    RHS[i] = RHS[i] + Integral_fphik
    RHS[i+1] = RHS[i+1] + Integral_fphik
  return RHS[1:-1]


In [ ]:
#@title discrete operator for Au =-$\nabla\cdot$(a(x) $\nabla$u)

# This function assembles the stiffness matrix for the operator Au = - div(a(x) grad(u)), using Gauss-Legendre quadrature.

def dsicrete_operator_div_a_grad_u(M, OMEGA, inner_triangles, a, quad_order = 3):

  l_boundary, r_boundary = OMEGA

  xi_q, w_q = leggauss(quad_order)

  tri = mesh(M, OMEGA)

  stiff = np.zeros((M+1,M+1))

  for i in range(M):

      x_1 = tri[i]
      x_2 = tri[i+1]
      h = x_2 -x_1
      grad_phi_1 = -1/h
      grad_phi_2 = 1/h
      local_stiff = np.zeros((2,2))

      for xi, w in zip(xi_q,w_q):

        xq = (x_2+x_1)/2 + h * (1/2) * xi

        local_stiff[0,0] += w * (grad_phi_1 * grad_phi_1 * a(xq)) * h * 1/2
        local_stiff[0,1] += w * (grad_phi_1 * grad_phi_2 * a(xq)) * h * 1/2
        local_stiff[1,0] += w * (grad_phi_2 * grad_phi_1 * a(xq)) * h * 1/2
        local_stiff[1,1] += w * (grad_phi_2 * grad_phi_2 * a(xq)) * h * 1/2

      stiff[i,i] += local_stiff[0,0]
      stiff[i,i+1] += local_stiff[0,1]
      stiff[i+1,i] += local_stiff[1,0]
      stiff[i+1,i+1] += local_stiff[1,1]

  stiff = stiff[1:-1,1:-1]

  return stiff



In [ ]:
#@title Parareal functions
#parareal functions, CP/FP, computing propagators sequentially, Test/error functions to varify results, caluclating right hand side
#functions to prepare using CP/FP or parareal itself to save calculations, by calculating values that occur multiple times beforehand.


#calculates the L2-error of the FEM solution U and the exact solution u(.,t), using Gauss-Legendre quadrature

def L2_error(M, OMEGA, U, u, t, quad_order = 5):
  l_boundary, r_boundary = OMEGA

  U_full = np.zeros(M + 1)
  U_full[1:-1] = U

  xi_q, w_q = leggauss(quad_order)

  tri = mesh(M, OMEGA)

  Integral_phi_u = 0

  for i in range(M):
    x_1 = tri[i]
    x_2 = tri[i+1]
    h = x_2 -x_1

    for xi, w in zip(xi_q,w_q):

      xq = (x_2+x_1)/2 + h * (1/2) * xi

      phi_1 = (x_2-xq)/h
      phi_2 = (xq-x_1)/h

      Integral_phi_u += w * ((phi_1 * U_full[i] + phi_2 * U_full[i+1]) - u(xq,t))**2 * (1/2) * h

  return np.sqrt(Integral_phi_u)


# solving M * f_h = F(t).
def f_h(t):
  RHS = calculate_RHS(f, mesh, M, OMEGA, t)
  return solve(mass, RHS)


# computes R(delta_T * A_h) and P_i(delta_T * A_h) beforehand. This is done because, for the propagators we use, these
# matrices dont change for fixed operator, propagator and time-grid.

def prepare_CP(delta_T, A_h, P, R):
  B = delta_T * A_h
  P_i_sum = []
  R_T_A_h = R(B)
  for i in range(len(P)):
    P_i_sum.append(P[i](B))
  return P_i_sum, R_T_A_h



# The function defined right above was suboptimal for propagators with high degrees in R(s) and P_i(s),
# and problems where the eigenvaluesgot very high, due to round off erors. This problem first occured
# for the last example for rho = 10000 and the OCP with degrees n=2 and m=4. However this was a homogeneous problem,
# and therefore this next function was only done for homogeneous problems.
# It uses A_h = V * diag(lambda_j) * V^{-1}, and we evaluate
# R(delta_T * A_h) = V * diag(R(delta_T * lambda_j)) * V^{-1}

def matrix_from_scalar_function(phi, delta_T, eigenvalues, eigenvectors, eigenvectors_inverse):
  values = np.asarray(phi(delta_T * eigenvalues), dtype=np.float64)
  return (eigenvectors * values) @ eigenvectors_inverse


def prepare_R_spectral(delta_T, phi, eigenvalues, eigenvectors, eigenvectors_inverse):
  R_T_A_h = matrix_from_scalar_function(phi, delta_T, eigenvalues, eigenvectors, eigenvectors_inverse)
  return [], R_T_A_h


# prepare_rhs computes values for the spatially discrete right hand side M^{-1}F(t),
# at t_n + c_i * delta_t, befor using the propagators/ parareal. This is also done to save a lot of calculations.

def prepare_rhs(C, f, mass, delta_t, T_intervall, M, OMEGA):

    RHS_values = {}

    t_0, t_end = T_intervall
    N_steps = int(round((t_end - t_0) / delta_t))
    T_n = np.linspace(t_0, t_end, N_steps + 1)
    for tn in T_n[:-1]:
        for c in C:
            t = round(float(tn + c * delta_t), 14)
            if t not in RHS_values:
                RHS = calculate_RHS(f,mesh,M,OMEGA,t)
                RHS_values[t] = solve(mass, RHS)
    return RHS_value


# This function evaluates on step of CP or FP, depending on the inputs R, P and C that you give.
# One can plug in R, P and C from the propagator dicitonary in this repository, which contains FP´s and CP´s
# that have been used in the work.

def CP(RHS_values, R_T_A_h, P_i_sum, T_n, delta_T, v, f_h, R, P, C, A_h):
  # if we have a homogeneous equation, the sum in the coarse / fine propagator over the p_i is always zero.
  if RHS_values is None:
    return R_T_A_h @ v

  B = delta_T * A_h
  rhs_sum = 0

  for i in range(len(P)):
    t = round(float(T_n + C[i] * delta_T),14)
    rhs_sum += (P_i_sum[i] @ RHS_values[t])


  return R_T_A_h @ v + delta_T * rhs_sum

# This sequentially computes a propagator on every time step in a grid of a time intervall

def solve_on_grid(RHS_values, R_T_A_h, P_i_sum, CP, T_intervall, v_h, f_h, N, R, P, C, A_h):
  t_0, t_end = T_intervall
  delta_T = (t_end-t_0)/N
  T_n = np.linspace(t_0, t_end, N+1)
  grid_solutions = []
  grid_solutions.append(v_h)

  for i in range(N):
    grid_solutions.append(CP(RHS_values, R_T_A_h, P_i_sum, T_n[i], delta_T, grid_solutions[i], f_h, R, P, C, A_h))

  return grid_solutions



def calculate_U(RHS_values, R_T_A_h, P_i_sum, f, M, N, OMEGA, mesh, u_0, T, R, P, C, A_h, mass, v_h):

  l_boundary, r_boundary = OMEGA
  tri = mesh(M, OMEGA)
  inner_tri = tri[1:-1]
  t_0, t_end = T
  T_n = np.linspace(t_0, t_end, N+1)


  U = solve_on_grid(RHS_values, R_T_A_h, P_i_sum, CP, T, v_h, f_h, N, R, P, C, A_h)

  return U




In [ ]:
#@title parareal error function

# maximum L2-error for two FEM-functions defined with mass matrix
def L2_for_FEM(U1, U2, mass):
    errors = []
    for u1, u2 in zip(U1, U2):
        e = u1 - u2
        errors.append(np.sqrt(e.T @ mass @ e))
    return max(errors)


# This is the parareal algorithm, which also takes the fine solution at every time step of the coarse grid as an input,
# and then while using the parareal algorithm, simultniously computes the parareal-fine solution error after every parareal iteration


def parareal_fehlervergleich(RHS_values_coarse, RHS_values_fine, R_T_A_h_coarse, R_T_A_h_fine, P_i_sum_coarse, P_i_sum_fnie, CP, T_intervall, U_fine, M, N, J, u_0, f, R_C, R_F, P_C, P_F, C_C, C_F, K, A_h, stiff, mass, v_h):

  t_0, t_end =T_intervall
  T_n_coarse = np.linspace(t_0, t_end, int(N/J)+1)
  delta_T = (t_end - t_0)/(int(N/J))

  error_hist = []

  U = solve_on_grid(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, CP, T_intervall, v_h, f_h, int(N/J), R_C, P_C, C_C, A_h)
  for k in range(K):

    U_alt = U.copy()
    Fine_solutions = [v_h]

    for i in range(int(N/J)):
        U_n_j = U[i]
        U_n1_k = solve_on_grid(RHS_values_fine, R_T_A_h_fine, P_i_sum_fnie, CP, (T_n_coarse[i],T_n_coarse[i+1]), U_n_j, f_h, J, R_F, P_F, C_F, A_h)[-1]
        Fine_solutions.append(U_n1_k)

    for i in range(int(N/J)):
      U[i+1] = CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U[i], f_h, R_C, P_C, C_C, A_h) + Fine_solutions[i+1] - CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U_alt[i], f_h, R_C, P_C, C_C, A_h)

    err = L2_for_FEM(U,U_fine, mass)
    error_hist.append(err)

  return U, error_hist

In [ ]:
#@title OCP stability function

#importing the parameters for the OCP stability function, building the stability funtion, and building the functions P_i(s).
#At the moment only for one function P_i, since q=1 for every optimization we have done in the bachelorthesis.



# evaluates a polynomial at the matrix B
def matrix_polynomial(B, coeffs):
    I = np.eye(B.shape[0])
    result = coeffs[0] * I
    B_power = I

    for coeff in coeffs[1:]:
        B_power = B_power @ B
        result = result + coeff * B_power

    return result


# The input for this is the filename of the file that was produced by the OCP-algorithm,
# containing the information about the stability-fucntion.
# This is done because otherwise you would either have to build the OCP algorithm inside the parareal function, or
# if you printed out the parameters there would be additional round off errors.
# This function can compute the scalar version, which is used in matrix_from_scalar_function, as well as the matrix version of
# the stability function.

def load_OCP(filename):

    saved = torch.load(filename, map_location="cpu", weights_only=False)

    if isinstance(saved, list):
        saved = saved[0]

    a = saved["a"].detach().cpu().numpy().astype(np.float64)
    b = saved["b"].detach().cpu().numpy().astype(np.float64)

    def R_scalar(s):
        s = np.asarray(s, dtype=np.float64)
        numerator = np.polynomial.polynomial.polyval(s, a)
        denominator = np.polynomial.polynomial.polyval(s, b)
        return numerator / denominator

    def R(B):
        I = np.eye(B.shape[0])

        numerator   = matrix_polynomial(B, a)
        denominator = matrix_polynomial(B, b)

        return numerator @ solve(denominator, I)

    # Für deine jetzigen q=1-OCPs
    def P1(B):
        I = np.eye(B.shape[0])
        return solve(B, I - R(B))

    return {
        "R": R,
        "P": [P1],
        "C": [1],
        "R_scalar": R_scalar,
        "a": a,
        "b": b
    }


